# Finish queue, two cards: finish1031

Resumes, in this order on whichever card is free: pairlatlab60fp64 seed 2; w64s24lab60 seed 0; w64s24lab60 seed 1; pairlab60fp64 seed 2. Mounts nsanya/so3c-k64-seed2fp64, nsanya/so3c-k64-w64s24lab60-seeds01. T4 with
torch 2.11.0+cu128 pinned and asserted. Kernel budget 7.05 h.

In [ ]:
import subprocess, time
T0 = time.time()
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True,
                     text=True).stdout.strip())

In [ ]:
import base64, io, tarfile, pathlib
CODE_B64 = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE_B64)), mode="r:gz").extractall("/kaggle/working/repo")
print(sorted(p.name for p in pathlib.Path("/kaggle/working/repo").iterdir()))

In [ ]:
import subprocess, sys

def run(args, cwd="/kaggle/working/repo"):
    """Run a repo module, stream nothing, return (ok, tail-of-output)."""
    r = subprocess.run([sys.executable, "-m"] + args, cwd=cwd,
                       capture_output=True, text=True)
    out = r.stdout[-4000:]
    if r.returncode != 0:
        out += "\n--- stderr ---\n" + r.stderr[-3000:]
    print(out)
    return r.returncode == 0

In [ ]:
import subprocess, sys, time
WANT = "2.11.0+cu128"
name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout.strip()
print("device:", name)
assert "T4" in name, "paired with T4 runs only; got %r -- stopping before any GPU time is spent" % name

def torch_version():
    return subprocess.run([sys.executable, "-c", "import torch; print(torch.__version__)"],
                          capture_output=True, text=True).stdout.strip()

v = torch_version()
if v != WANT:
    print("torch", v, "- installing", WANT)
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.11.0",
                        "--index-url", "https://download.pytorch.org/whl/cu128"],
                       capture_output=True, text=True)
    print("pip rc", r.returncode, "in %.0f s" % (time.perf_counter() - t0))
    if r.returncode:
        print(r.stderr[-2000:])
        raise SystemExit("could not install torch " + WANT)
    v = torch_version()
assert v == WANT, "torch %s, expected %s" % (v, WANT)
print("torch", v, "on", name)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "torchdiffeq"], check=True)

In [ ]:
import os, subprocess, sys

def launch(args, gpu, log):
    """Start a repo module on one card; its output goes to `log`."""
    # unbuffered, with faulthandler: a killed or crashed child still leaves
    # its last lines and a traceback in `log`
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONFAULTHANDLER="1")
    return subprocess.Popen([sys.executable, "-m"] + args, cwd="/kaggle/working/repo",
                            env=env, stdout=open(log, "w"), stderr=subprocess.STDOUT)

def tail(log, n=3000):
    return open(log).read()[-n:]

cards = [l for l in subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                   capture_output=True, text=True).stdout.splitlines() if l.strip()]
NGPU = len(cards)
print("cards:", cards)

In [ ]:
import glob, pathlib
cands = glob.glob("/kaggle/input/**/top_tagging_train.npz", recursive=True)
assert cands, "attach the K=128 data-prep kernel output"
DATA = str(pathlib.Path(cands[0]).parent)
NAME, JOBS, EPOCHS, BUDGET = 'finish1031', [['pairlatlab60fp64', ['--keep-best', '--pair-lab', '--pair-latent', '8', '--pair-lab-fp64'], 2], ['w64s24lab60', ['--keep-best', '--pair-lab', '--act-hidden', '64', '--msg-dim', '24', '--scalar-dim', '24'], 0], ['w64s24lab60', ['--keep-best', '--pair-lab', '--act-hidden', '64', '--msg-dim', '24', '--scalar-dim', '24'], 1], ['pairlab60fp64', ['--keep-best', '--pair-lab', '--pair-lab-fp64'], 2]], 60, 7.05 * 3600
MARGIN = 0.40 * 3600          # one epoch's overrun and the final evaluation
OUT = "/kaggle/working/results_k64"
CKPT = "/kaggle/working/checkpoints"
LOGS = "/kaggle/working/logs/" + NAME
BASE = ['--cache-dir', 'DATA', '--representation', 'constituents', '--canonical-splits', '--normalize', 'global', '--device', 'cuda', '--dtype', 'float32', '--batch-size', '256', '--n-constituents', '64', '--eval-chunk-size', '1024', '--models', 'so3c_message_set', '--rounds', '3', '--beams', '--channels', '8', '--compile']
BASE[BASE.index("DATA")] = DATA
pathlib.Path(LOGS).mkdir(parents=True, exist_ok=True)
for tag, _, _ in JOBS:
    pathlib.Path(OUT, tag).mkdir(parents=True, exist_ok=True)
    pathlib.Path(OUT, tag, "bundle_sha256.txt").write_text('1f0ff82417b0' + chr(10))
print("data:", DATA, "| bundle", '1f0ff82417b0', "| budget %.2f h" % (BUDGET / 3600))

In [ ]:
# On-card guard: the forward pass must be the published function on this
# GPU too, in float64. "1 passed" is required; a skip means no CUDA.
import subprocess, sys
g = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                    "tests/test_so3c_models.py", "-k", "golden_cuda"],
                   cwd="/kaggle/working/repo", capture_output=True, text=True)
print(g.stdout[-1500:], g.stderr[-1500:])
assert "1 passed" in g.stdout, "golden guard did not pass on the card"

In [ ]:
# The first kernels' last checkpoints, from their mounted outputs. Each must
# agree with the result JSON its kernel wrote (epoch, batch order) and goes
# where --resume looks; the result JSON does not, so the run is not skipped.
import glob, json, pathlib, shutil, torch
TODO, DONE = [], {}
for tag, flags, s in JOBS:
    pat = "/kaggle/input/**/%s/" + tag + "/seed%d/*seed%d.%s"
    cks = glob.glob(pat % ("checkpoints", s, s, "pt"), recursive=True)
    js = glob.glob(pat % ("results_k64", s, s, "json"), recursive=True)
    assert len(cks) == 1 and len(js) == 1, (tag, s, cks, js)
    st = torch.load(cks[0], map_location="cpu", weights_only=False)
    rec = json.load(open(js[0]))
    print("%s seed %d: checkpoint at epoch %d (history %d), first kernel's result at epoch %d"
          % (tag, s, st["epoch"], len(st["history"]), rec["epochs_run"]))
    assert len(st["history"]) == st["epoch"] == rec["epochs_run"], (tag, s)
    assert rec["batch_order"]["epoch1"] == st["batch_order_epoch1"], (tag, s)
    if st["epoch"] >= EPOCHS:
        DONE[(tag, s)] = rec
        continue
    ck = pathlib.Path(CKPT, tag, "seed%d" % s, pathlib.Path(cks[0]).name)
    ck.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(cks[0], ck)
    TODO.append((tag, flags, s))
    del st
print("queue:", [(t, s) for t, _, s in TODO], "| finished in the first kernels:", sorted(DONE))

In [ ]:
# The queue: each free card takes the next job, with --max-seconds from what
# is left of the budget; no job starts with less than half an hour left.
import glob, time
todo = list(TODO)
running = {}                      # card -> (tag, seed, process)
BAD, SKIPPED = [], []
while todo or running:
    for c in range(NGPU):
        if c in running:
            tag, s, p = running[c]
            if p.poll() is None:
                continue
            ok = p.returncode == 0 and glob.glob(OUT + "/%s/seed%d/*.json" % (tag, s))
            print("%7.2f h  card %d: %s seed %d %s" % ((time.time() - T0) / 3600, c, tag, s,
                                                       "done" if ok else "FAILED, exit %d" % p.returncode), flush=True)
            if not ok:
                BAD.append((tag, s)); print(tail(LOGS + "/%s_seed%d.log" % (tag, s), 1500))
            del running[c]
        if todo and c not in running:
            left = int(BUDGET - (time.time() - T0) - MARGIN)
            if left < 1800:
                SKIPPED += [(t, s) for t, _, s in todo]; todo = []
                print("budget: the remaining jobs are skipped:", SKIPPED); continue
            tag, flags, s = todo.pop(0)
            running[c] = (tag, s, launch(["benchmarks.run_top_tagging"] + BASE + flags +
                                         ["--epochs", str(EPOCHS), "--seed", str(s),
                                          "--results-dir", OUT + "/%s/seed%d" % (tag, s),
                                          "--ckpt-dir", CKPT + "/%s/seed%d" % (tag, s),
                                          "--resume", "--max-seconds", str(left)],
                                         c, LOGS + "/%s_seed%d.log" % (tag, s)))
            print("%7.2f h  card %d: start %s seed %d (max %.2f h)" % ((time.time() - T0) / 3600, c, tag, s, left / 3600), flush=True)
    time.sleep(20)
print("failed:", BAD or "none", "| skipped:", SKIPPED or "none")
print("kernel time %.2f h" % ((time.time() - T0) / 3600))

In [ ]:
# Both K=64 readings of the window, by the rule fixed before the push
# (K64_CONFIRM_RULE): confirmed if the arm's mean gains >= +0.0003 AUC on its
# reference and every arm run lies above every reference run; not confirmed
# if it gains <= +0.0000; otherwise unresolved. A float is a committed run's
# test AUC; [tag, seed] a run of this window.
import json, glob, math, statistics
READINGS, RULE = [('pair latent at K=64, three seeds', [0.9846450602988644, 0.9849288396121384, ['pairlatlab60fp64', 2]], [0.9841706950893652, 0.9846654727988882, ['pairlab60fp64', 2]]), ('width at K=64', [['w64s24lab60', 0], ['w64s24lab60', 1]], [0.9841706950893652, 0.9846654727988882])], (0.0003, 0.0)
POOLED_SD, POOLED_DF = (0.00028969561359768256, 2)   # within-configuration sd of base60 and pairlab60
mean = statistics.mean
recs = dict(DONE)
for tag, _, s in JOBS:
    for f in glob.glob(OUT + "/%s/seed%d/*.json" % (tag, s)):
        recs[(tag, s)] = json.load(open(f))
for (tag, s), r in sorted(recs.items()):
    t = r["test_metrics"]; env = r["env"]; h = r["history"]; vl = [e["val_loss"] for e in h]
    assert env["torch"] == "2.11.0+cu128" and "T4" in env.get("gpu", ""), env
    print("%s seed %d: %d epochs (history %d), %.2f h cumulative, %d params | kwargs %s | compile %s | batch order %s"
          % (tag, s, r["epochs_run"], len(h), r["walltime_sec"] / 3600, r["n_params"], r["so3c_kwargs"],
             (r.get("compile_check") or {}).get("used"), r["batch_order"]["epoch1"][:12]))
    print("  AUC %.5f  rej@0.3 %.1f  rej@0.5 %.1f  final val loss %.5f (lowest %.5f at epoch %d)  val - train %+.5f"
          % (t["test_auc"], t["bg_rej_30"], t.get("bg_rej_50", float("nan")), vl[-1], min(vl),
             h[vl.index(min(vl))]["epoch"], vl[-1] - h[-1]["train_loss"]))
    if r["epochs_run"] < EPOCHS:
        print("  INCOMPLETE (epoch %d of %d): stopped on the budget; resume it next window" % (r["epochs_run"], EPOCHS))

def value(item):
    if isinstance(item, float):
        return item
    r = recs.get(tuple(item))
    return r["test_metrics"]["test_auc"] if r and r["epochs_run"] >= EPOCHS else None

def p_two_sided(t, df):
    try:
        from scipy import stats
        return 2 * stats.t.sf(abs(t), df)
    except Exception:
        return float("nan")

for name, arm, ref in READINGS:
    a, b = [value(x) for x in arm], [value(x) for x in ref]
    if None in a or None in b:
        print("%s: no verdict, runs incomplete" % name)
        continue
    d = mean(a) - mean(b)
    tp = d / (POOLED_SD * math.sqrt(1 / len(a) + 1 / len(b)))
    if d >= RULE[0] and min(a) > max(b):
        verdict = "CONFIRMED at K=64"
    elif d <= RULE[1]:
        verdict = "NOT CONFIRMED"
    else:
        verdict = "unresolved"
    print("%s: arm %s mean %.5f | reference %s mean %.5f | d %+.5f | against the pooled sd %.5f: t %.2f, df %d, p %.3f | arm runs above every reference run: %d of %d -> %s"
          % (name, " ".join("%.5f" % x for x in a), mean(a), " ".join("%.5f" % x for x in b), mean(b), d,
             POOLED_SD, tp, POOLED_DF, p_two_sided(tp, POOLED_DF), sum(x > max(b) for x in a), len(a), verdict))
pl, rf = value(["pairlatlab60fp64", 2]), value(["pairlab60fp64", 2])
if pl is not None and rf is not None:
    print("seed-2 pair (float64 features, started together): pair latent %+.5f" % (pl - rf))
assert not BAD, "jobs failed: %s" % BAD